<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

В данной версии к существующей иерархии добавлены:
1. Коллекции (List<T>, Dictionary<TKey, TValue>): Реализовано хранение истории заказов, списка доступных опций доставки и быстрого поиска по трек-номеру через словарь.
2. Делегаты и события: Добавлены события изменения статуса доставки и отправки уведомлений клиенту с использованием специализированных делегатов.
3. Расширение свойств и методов: К базовым и производным классам добавлены новые атрибуты (интервалы времени, тарифы, статусы) и методы управления заказами.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Определение делегата для событий доставки
public delegate void DeliveryEventHandler(string message);

public class Order
{
    private int _orderId;
    private double _packageWeight;
    private string _receiverName;
    private decimal _declaredValue;
    private bool _isFragile;

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value > 0 ? value : 1; }
    }

    public double PackageWeight
    {
        get { return _packageWeight; }
        set { _packageWeight = value > 0 ? value : 0.1; }
    }

    public string ReceiverName
    {
        get { return _receiverName; }
        set { _receiverName = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal DeclaredValue
    {
        get { return _declaredValue; }
        set { _declaredValue = value >= 0 ? value : 0; }
    }

    public bool IsFragile
    {
        get { return _isFragile; }
        set { _isFragile = value; }
    }

    public Order(int orderId, double packageWeight, string receiverName, decimal declaredValue, bool isFragile = false)
    {
        OrderId = orderId;
        PackageWeight = packageWeight;
        ReceiverName = receiverName;
        DeclaredValue = declaredValue;
        IsFragile = isFragile;
    }

    public string GetOrderSummary()
    {
        return $"Order #{OrderId} | Receiver: {ReceiverName} | Weight: {PackageWeight}kg | Value: {DeclaredValue:C} | Fragile: {IsFragile}";
    }

    public void UpdateWeight(double newWeight)
    {
        PackageWeight = newWeight;
    }
}

// Интерфейсы
public interface ITrackable
{
    string TrackingNumber { get; set; }
    string GetTrackingStatus();
    void UpdateStatus(string status);
}

public interface IInsurable
{
    decimal InsuranceRate { get; set; }
    decimal CalculateInsuranceCost(Order order);
}

// Базовый класс ShippingOption
public class ShippingOption
{
    private int _id;
    private string _name;
    private decimal _baseCost;
    private double _maxWeight;
    private string _description;
    private bool _isActive;
    private int _priorityLevel;
    private decimal _handlingFee;

    // СОБЫТИЕ изменения статуса опции доставки
    public event DeliveryEventHandler StatusChanged;

    public int Id
    {
        get { return _id; }
        set { _id = value; }
    }

    public string Name
    {
        get { return _name; }
        set { _name = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal BaseCost
    {
        get { return _baseCost; }
        set { _baseCost = value >= 0 ? value : 0; }
    }

    public double MaxWeight
    {
        get { return _maxWeight; }
        set { _maxWeight = value > 0 ? value : 1000; }
    }

    public string Description
    {
        get { return _description; }
        set { _description = string.IsNullOrWhiteSpace(value) ? "-" : value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public int PriorityLevel
    {
        get { return _priorityLevel; }
        set { _priorityLevel = value > 0 ? value : 1; }
    }

    public decimal HandlingFee
    {
        get { return _handlingFee; }
        set { _handlingFee = value >= 0 ? value : 0; }
    }

    public ShippingOption(int id, string name, decimal baseCost, double maxWeight)
    {
        Id = id;
        Name = name;
        BaseCost = baseCost;
        MaxWeight = maxWeight;
        IsActive = true;
        Description = "Standard shipping option";
        PriorityLevel = 1;
        HandlingFee = 50m;
    }

    public virtual decimal CalculateCost(Order order)
    {
        return BaseCost + (order.PackageWeight > 1.0 ? (decimal)(order.PackageWeight - 1.0) * 50m : 0m);
    }

    public virtual string GetDeliveryDetails()
    {
        return $"[{Id}] {Name} | Base Cost: {BaseCost:C} | Max Weight: {MaxWeight}kg";
    }

    public bool CheckAvailability(Order order)
    {
        return IsActive && order.PackageWeight <= MaxWeight;
    }

    public void ApplyDiscount(decimal percent)
    {
        if (percent > 0 && percent <= 100)
        {
            BaseCost -= BaseCost * (percent / 100);
            OnStatusChanged($"Discount of {percent}% applied to option '{Name}'.");
        }
    }

    public void ToggleActiveStatus()
    {
        IsActive = !IsActive;
        OnStatusChanged($"Active status for option '{Name}' changed to: {IsActive}");
    }

    protected void OnStatusChanged(string message)
    {
        StatusChanged?.Invoke(message);
    }
}

// Промежуточный класс CourierDelivery
public class CourierDelivery : ShippingOption
{
    private string _courierName;
    private string _vehicleType;
    private string _contactPhone;
    private bool _isContactlessAllowed;

    public string CourierName
    {
        get { return _courierName; }
        set { _courierName = string.IsNullOrWhiteSpace(value) ? "Pending" : value; }
    }

    public string VehicleType
    {
        get { return _vehicleType; }
        set { _vehicleType = string.IsNullOrWhiteSpace(value) ? "Van" : value; }
    }

    public string ContactPhone
    {
        get { return _contactPhone; }
        set { _contactPhone = string.IsNullOrWhiteSpace(value) ? "N/A" : value; }
    }

    public bool IsContactlessAllowed
    {
        get { return _isContactlessAllowed; }
        set { _isContactlessAllowed = value; }
    }

    public CourierDelivery(int id, string name, decimal baseCost, double maxWeight) 
        : base(id, name, baseCost, maxWeight)
    {
        CourierName = "Not assigned";
        VehicleType = "Van";
        ContactPhone = "000-000";
        IsContactlessAllowed = true;
    }

    public void AssignCourier(string name, string phone)
    {
        CourierName = name;
        ContactPhone = phone;
        OnStatusChanged($"Courier {name} assigned to delivery.");
    }

    public string GetCourierInfo()
    {
        return $"Courier: {CourierName} | Phone: {ContactPhone} | Vehicle: {VehicleType}";
    }

    public void UpgradeVehicle(string newVehicle)
    {
        VehicleType = newVehicle;
        OnStatusChanged($"Vehicle upgraded to {newVehicle}.");
    }
}

// Класс ExpressDelivery
public class ExpressDelivery : CourierDelivery, ITrackable, IInsurable
{
    private int _minTime;
    private decimal _surcharge;
    private string _trackingNumber;
    private decimal _insuranceRate;
    private string _currentStatus;

    public int MinTime
    {
        get { return _minTime; }
        set { _minTime = value > 0 ? value : 1; }
    }

    public decimal Surcharge
    {
        get { return _surcharge; }
        set { _surcharge = value >= 0 ? value : 0; }
    }

    public ExpressDelivery(int id, string name, decimal baseCost, double maxWeight, int minTime, decimal surcharge) 
        : base(id, name, baseCost, maxWeight)
    {
        MinTime = minTime;
        Surcharge = surcharge;
        _trackingNumber = "EXP-DEFAULT";
        _insuranceRate = 0.05m;
        _currentStatus = "Created";
    }

    string ITrackable.TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = value; }
    }

    string ITrackable.GetTrackingStatus()
    {
        return $"[EXPRESS TRACKING #{_trackingNumber}] Status: {_currentStatus}";
    }

    void ITrackable.UpdateStatus(string status)
    {
        _currentStatus = status;
        OnStatusChanged($"Express status updated to: {status}");
    }

    decimal IInsurable.InsuranceRate
    {
        get { return _insuranceRate; }
        set { _insuranceRate = value; }
    }

    decimal IInsurable.CalculateInsuranceCost(Order order)
    {
        return order.DeclaredValue * _insuranceRate;
    }

    public override decimal CalculateCost(Order order)
    {
        decimal insurance = ((IInsurable)this).CalculateInsuranceCost(order);
        return base.CalculateCost(order) + Surcharge + insurance;
    }
}

// Класс Pickup
public class Pickup : ShippingOption, ITrackable
{
    private string _address;
    private int _storageDays;
    private string _trackingNumber;
    private string _currentStatus;

    public string Address
    {
        get { return _address; }
        set { _address = string.IsNullOrWhiteSpace(value) ? "Main Hub" : value; }
    }

    public int StorageDays
    {
        get { return _storageDays; }
        set { _storageDays = value > 0 ? value : 3; }
    }

    public Pickup(int id, string name, decimal baseCost, double maxWeight, string address) 
        : base(id, name, baseCost, maxWeight)
    {
        Address = address;
        StorageDays = 5;
        _trackingNumber = "PKP-DEFAULT";
        _currentStatus = "Awaiting arrival";
    }

    string ITrackable.TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = value; }
    }

    string ITrackable.GetTrackingStatus()
    {
        return $"[PICKUP TRACKING #{_trackingNumber}] Status: {_currentStatus} at {Address}";
    }

    void ITrackable.UpdateStatus(string status)
    {
        _currentStatus = status;
        OnStatusChanged($"Pickup status updated to: {status}");
    }

    public override decimal CalculateCost(Order order)
    {
        return BaseCost;
    }

    public void ExtendStorage(int days)
    {
        if (days > 0)
        {
            StorageDays += days;
            OnStatusChanged($"Storage extended by {days} days.");
        }
    }
}

// Менеджер логистики с коллекциями (List<T> и Dictionary<TKey, TValue>)
public class LogisticsHub
{
    // Список всех доступных опций доставки (List<T>)
    private List<ShippingOption> _optionsList = new List<ShippingOption>();

    // Словарь для быстрого поиска опции трекинга по TrackingNumber (Dictionary<string, ITrackable>)
    private Dictionary<string, ITrackable> _trackingDictionary = new Dictionary<string, ITrackable>();

    public void RegisterOption(ShippingOption option)
    {
        _optionsList.Add(option);
        if (option is ITrackable trackable)
        {
            _trackingDictionary[trackable.TrackingNumber] = trackable;
        }
    }

    public void DisplayAllOptions(Order order)
    {
        Console.WriteLine($"\n--- Available Shipping Options for Order #{order.OrderId} ---");
        foreach (var option in _optionsList)
        {
            if (option.CheckAvailability(order))
            {
                Console.WriteLine($"{option.Name} -> Cost: {option.CalculateCost(order):C}");
            }
        }
    }

    public void TrackByNumber(string trackingNumber)
    {
        if (_trackingDictionary.TryGetValue(trackingNumber, out ITrackable trackable))
        {
            Console.WriteLine($"\n[Quick Search Result]: {trackable.GetTrackingStatus()}");
        }
        else
        {
            Console.WriteLine($"\n[Quick Search Result]: Tracking number '{trackingNumber}' not found.");
        }
    }
}

public class Program
{
    // Обработчик системных событий
    static void GlobalEventHandler(string message)
    {
        Console.ForegroundColor = ConsoleColor.Cyan;
        Console.WriteLine($"[EVENT LOG]: {message}");
        Console.ResetColor();
    }

    public static void Main()
    {
        Order order1 = new Order(3001, 2.0, "Ivan Ivanov", 5000m, false);

        ExpressDelivery express = new ExpressDelivery(10, "Super Express", 500m, 10.0, 12, 200m);
        ((ITrackable)express).TrackingNumber = "EXP-TRACK-001";
        
        // Подписка на событие изменения статуса опции доставки
        express.StatusChanged += GlobalEventHandler;

        Pickup pickup = new Pickup(20, "Local Post", 100m, 25.0, "Central St. 5");
        ((ITrackable)pickup).TrackingNumber = "PKP-TRACK-002";
        pickup.StatusChanged += GlobalEventHandler;

        // Создаем хаб логистики с коллекциями
        LogisticsHub hub = new LogisticsHub();
        hub.RegisterOption(express);
        hub.RegisterOption(pickup);

        // Демонстрация работы списка List<T>
        hub.DisplayAllOptions(order1);

        Console.WriteLine("\n--- Triggering Actions & Events ---");
        express.AssignCourier("Alex", "+7-555-0199");
        express.ApplyDiscount(15);

        pickup.ExtendStorage(3);

        // Демонстрация работы словаря Dictionary<TKey, TValue>
        hub.TrackByNumber("EXP-TRACK-001");
        hub.TrackByNumber("UNKNOWN-NUM");
    }
}